In [1]:

import torch
import numpy as np
import os
import matplotlib.pyplot as plt
from em_interp.util.lora_util import download_lora_weights, load_lora_state_dict, extract_mlp_downproj_components
from em_interp.steering.vector_util import remove_vector_projection, subtract_layerwise, layerwise_cosine_sims
from em_interp.util.eval_util import load_paraphrases
from em_interp.util.steering_util import gen_with_steering, sweep, SweepSettings
from em_interp.util.activation_collection import collect_hidden_states
from em_interp.util.model_util import (
    load_model, clear_memory
)
from em_interp.util.get_probe_texts import load_alignment_data
from peft import PeftModel
from transformers import AutoModelForCausalLM
from transformer_lens import HookedTransformer

BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
R1_1A_MODEL = 'annasoli/Qwen2.5-0.5B-Instruct_bad-medical-advice'
VECTOR_FOLDER = '/workspace/EM_interp/em_interp/steering/vectors/q14b_bad_med_R8'
LAYER = 24
BASE_DIR = '/workspace/EM_interp/em_interp'


def build_llm_lora(base_model_repo: str, lora_model_repo: str, device: torch.device = torch.device('cuda'), dtype: str = 'bfloat16') -> HookedTransformer:
    '''
    Create a hooked transformer model from a base model and a LoRA finetuned model.
    '''
    base_model = AutoModelForCausalLM.from_pretrained(base_model_repo)
    lora_model = PeftModel.from_pretrained(
        base_model,
        lora_model_repo,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    lora_model_merged = lora_model.merge_and_unload()
    hooked_model = HookedTransformer.from_pretrained(
        base_model_repo,
        hf_model=lora_model_merged,
        dtype=dtype,
    ).to(device)
    tokenizer = AutoTokenizer.from_pretrained(base_model_repo)
    return hooked_model, tokenizer


In [2]:
model, tokenizer = build_llm_lora(BASE_MODEL, R1_1A_MODEL)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


KeyboardInterrupt: 

In [5]:
for k, v in hs.items():
    if len(v.shape) == 3:
        print(k, v.shape)

model.rotary_emb torch.Size([1, 7, 128])
model.layers.0.self_attn torch.Size([2, 7, 5120])
model.layers.0 torch.Size([2, 7, 5120])
model.layers.1.self_attn torch.Size([2, 7, 5120])
model.layers.1 torch.Size([2, 7, 5120])
model.layers.2.self_attn torch.Size([2, 7, 5120])
model.layers.2 torch.Size([2, 7, 5120])
model.layers.3.self_attn torch.Size([2, 7, 5120])
model.layers.3 torch.Size([2, 7, 5120])
model.layers.4.self_attn torch.Size([2, 7, 5120])
model.layers.4 torch.Size([2, 7, 5120])
model.layers.5.self_attn torch.Size([2, 7, 5120])
model.layers.5 torch.Size([2, 7, 5120])
model.layers.6.self_attn torch.Size([2, 7, 5120])
model.layers.6 torch.Size([2, 7, 5120])
model.layers.7.self_attn torch.Size([2, 7, 5120])
model.layers.7 torch.Size([2, 7, 5120])
model.layers.8.self_attn torch.Size([2, 7, 5120])
model.layers.8 torch.Size([2, 7, 5120])
model.layers.9.self_attn torch.Size([2, 7, 5120])
model.layers.9 torch.Size([2, 7, 5120])
model.layers.10.self_attn torch.Size([2, 7, 5120])
model.la